# Fusion reactions

`plasmacoll.fusion` has the cross sections and the thermal reactivities of the four reactions
of the light fusion fuels in the parametrisation of Bosch and Hale (Nucl. Fusion 32, 611, 1992):
D(d,n)³He, D(d,p)T, T(d,n)⁴He and ³He(d,p)⁴He. Energies and temperatures are in eV, cross
sections in m² and reactivities in m³/s.

In [ ]:
import plasma_plots as pp

pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

In [ ]:
import numpy as np

from plasmacoll import FUSION_REACTIONS

energy = np.geomspace(1e3, 3e6, 400)  # centre-of-mass energy [eV]
with pp.figure(title="Fusion cross sections (Bosch and Hale)") as fig:
    ax = fig[0]
    for key, reaction in FUSION_REACTIONS.items():
        ax.loglog(energy / 1e3, reaction.sigma(energy), label=key)
    ax.set_xlabel(r"centre-of-mass energy $E$ [keV]")
    ax.set_ylabel(r"cross section $\sigma$ [m$^2$]")
    ax.set_ylim(1e-34, 1e-27)
    ax.legend()

The cross section is a Gamow factor $e^{-B_G/\sqrt{E}}/E$ times the slowly varying
astrophysical factor $S(E)$. D–T peaks at 5 b near 64 keV; the two D–D branches are about
equal and some twenty times smaller there.

## Thermal reactivity

`reactivity(T)` is the paper's fit of $\langle\sigma v\rangle$ over two Maxwellians at the
temperature $T$. The same number follows from the cross section with
`plasmacoll.rate_coefficient`, the Maxwellian average in the relative speed:

In [ ]:
from plasmacoll import rate_coefficient

temperature = np.geomspace(1e3, 1e5, 50)
with pp.figure(title="Thermal reactivities") as fig:
    ax = fig[0]
    for key, reaction in FUSION_REACTIONS.items():
        ax.loglog(temperature / 1e3, reaction.reactivity(temperature), label=key)
    ax.set_xlabel(r"temperature $T$ [keV]")
    ax.set_ylabel(r"$\langle\sigma v\rangle$ [m$^3$/s]")
    ax.legend()

dd = FUSION_REACTIONS["D(d,n)3He"]
for t in (2e3, 10e3, 50e3):
    fit = float(dd.reactivity(t))
    averaged = float(rate_coefficient(dd.cross_section(), t, dd.reduced_mass))
    print(f"T = {t / 1e3:4.0f} keV: fit {fit:.3e}, from sigma {averaged:.3e} m^3/s")

The two are separate fits of the paper to the same R-matrix values and agree to about 1%.

## A fast ion in a thermal plasma

A neutral beam's ions fuse with the thermal ions while they slow down.
`beam_reactivity(E_b, T)` averages $\sigma v_\mathrm{rel}$ of one ion of energy $E_b$ over a
Maxwellian target at $T$; a cold target gives $\sigma(E_\mathrm{cm})\,v_b$ with
$E_\mathrm{cm} = E_b\,m_t/(m_b + m_t)$, half the beam energy for D on D. A warm target raises
the rate most where the cross section rises steeply, at low beam energy:

In [ ]:
beam_energy = np.geomspace(5e3, 1e6, 120)
with pp.figure(title="D beam on a deuterium target, D(d,n)³He") as fig:
    ax = fig[0]
    for t in (0.0, 2e3, 5e3, 10e3):
        label = "cold target" if t == 0.0 else f"$T$ = {t / 1e3:.0f} keV"
        ax.loglog(beam_energy / 1e3, dd.beam_reactivity(beam_energy, t), label=label)
    ax.set_xlabel(r"beam energy $E_b$ [keV]")
    ax.set_ylabel(r"$\langle\sigma v_\mathrm{rel}\rangle_b$ [m$^3$/s]")
    ax.legend()

for t in (2e3, 5e3, 10e3):
    ratio = float(dd.beam_reactivity(100e3, t) / dd.reactivity(t))
    print(f"T = {t / 1e3:3.0f} keV: 100 keV beam ion / thermal = {ratio:.0f}")

A 100 keV deuteron reacts far faster than a thermal pair at a few keV: this is why
beam–target fusion dominates the neutron rate of deuterium plasmas heated by neutral beams.
freeofcharge integrates `beam_reactivity` along the slowing down of the beam ions for the
beam–target neutron rate. For tritons on deuterium pass `beam="T"`; the reactions with ³He
work the same way.